# Notebook 5: Mendelian Randomisation — Independent Causal Evidence
## CausalPMOS-Map — K(g) Component

**Goal:** Use Mendelian Randomisation (MR) and GWAS catalog evidence to
compute the **Independent Knowledge component K(g)** of the PCBS score.
K(g) captures genetic/causal evidence that a gene sits on the PMOS → EC
pathway, independent of the expression analyses in NB1–NB4.

**Methods:**
- **Bidirectional two-sample MR** — PCOS → EC (forward) and EC → PCOS (reverse)
  using 4 complementary estimators: IVW, MR-Egger, Weighted Median, Simple Mode
- **Instruments:** Genome-wide significant loci from published GWAS meta-analyses
  (Day et al. 2018 for PCOS; O’Mara et al. 2018 for EC)
- **GWAS Catalog gene lists** — genes with genome-wide significant associations for
  PCOS and/or EC, curated from published meta-analyses
- **K(g) scoring** — per-gene score ∈ [0, 1] based on MR instrument proximity,
  GWAS trait associations, and shared cross-trait evidence

**Data sources:**
- Day et al. 2018 (Nat Genet) — PCOS GWAS meta-analysis (10,074 cases)
- O’Mara et al. 2018 (Nat Genet) — EC GWAS meta-analysis (12,906 cases)
- NHGRI-EBI GWAS Catalog (gene–trait associations)

**Key output:** `results/kg_scores_mr.csv`

## 1. Setup

In [ ]:
# ── 1 · Install packages & imports ─────────────────────────────
import warnings, os
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns

os.makedirs('results', exist_ok=True)
os.makedirs('figures', exist_ok=True)

print('Setup complete.')

## 2. MR Method Implementations

Four complementary two-sample MR estimators, each with different
assumptions about instrument validity:

| Method | Assumption | Strength |
|--------|-----------|----------|
| **IVW** | All instruments valid (no pleiotropy) | Most powerful |
| **MR-Egger** | InSIDE assumption; tests for directional pleiotropy | Intercept = pleiotropy test |
| **Weighted Median** | ≥50% of instruments valid | Robust to outliers |
| **Simple Mode** | Plurality of instruments valid | Most conservative |

Agreement across methods strengthens causal inference.

In [ ]:
# ── 2 · MR method implementations ─────────────────────────────

def mr_ivw(beta_exp, beta_out, se_out):
    """Inverse-variance weighted MR (fixed effects, second-order weights)."""
    w = (beta_exp**2) / (se_out**2)
    wald = beta_out / beta_exp
    beta_ivw = np.sum(w * wald) / np.sum(w)
    se_ivw = np.sqrt(1 / np.sum(w))
    p = 2 * stats.norm.sf(abs(beta_ivw / se_ivw))

    # Cochran's Q for heterogeneity
    q = np.sum(w * (wald - beta_ivw)**2)
    q_df = len(beta_exp) - 1
    q_p = 1 - stats.chi2.cdf(q, q_df) if q_df > 0 else np.nan
    i2 = max(0, (q - q_df) / q * 100) if q > 0 else 0

    return {'method': 'IVW', 'beta': beta_ivw, 'se': se_ivw, 'pval': p,
            'OR': np.exp(beta_ivw),
            'CI_lo': np.exp(beta_ivw - 1.96 * se_ivw),
            'CI_hi': np.exp(beta_ivw + 1.96 * se_ivw),
            'Q': q, 'Q_pval': q_p, 'I2': i2, 'n_snps': len(beta_exp)}


def mr_egger(beta_exp, beta_out, se_out):
    """MR-Egger regression (tests for directional pleiotropy)."""
    if len(beta_exp) < 3:
        return {'method': 'MR-Egger', 'beta': np.nan, 'se': np.nan,
                'pval': np.nan, 'OR': np.nan, 'CI_lo': np.nan,
                'CI_hi': np.nan, 'intercept': np.nan,
                'intercept_pval': np.nan, 'n_snps': len(beta_exp)}

    # Orient so exposure beta is positive
    sign = np.sign(beta_exp)
    bx = np.abs(beta_exp)
    by = beta_out * sign
    w = 1 / se_out**2

    X = sm.add_constant(bx)
    model = sm.WLS(by, X, weights=w).fit()

    intercept = model.params[0]
    beta_eg = model.params[1]
    se_eg = model.bse[1]
    p_eg = model.pvalues[1]
    p_int = model.pvalues[0]

    return {'method': 'MR-Egger', 'beta': beta_eg, 'se': se_eg, 'pval': p_eg,
            'OR': np.exp(beta_eg),
            'CI_lo': np.exp(beta_eg - 1.96 * se_eg),
            'CI_hi': np.exp(beta_eg + 1.96 * se_eg),
            'intercept': intercept, 'intercept_pval': p_int,
            'n_snps': len(beta_exp)}


def mr_weighted_median(beta_exp, beta_out, se_out, n_boot=1000):
    """Weighted median MR estimator (valid if >=50% instruments valid)."""
    wald = beta_out / beta_exp
    se_wald = np.abs(se_out / beta_exp)
    weights = 1 / se_wald**2

    def _weighted_median(vals, wts):
        order = np.argsort(vals)
        vals_s = vals[order]
        wts_s = wts[order]
        cum = np.cumsum(wts_s) / wts_s.sum()
        idx = np.searchsorted(cum, 0.5)
        return vals_s[min(idx, len(vals_s) - 1)]

    beta_wm = _weighted_median(wald, weights)

    np.random.seed(42)
    betas_boot = []
    for _ in range(n_boot):
        i = np.random.choice(len(wald), len(wald), replace=True)
        betas_boot.append(_weighted_median(wald[i], weights[i]))
    se_wm = np.std(betas_boot)
    p_wm = 2 * stats.norm.sf(abs(beta_wm / se_wm)) if se_wm > 0 else np.nan

    return {'method': 'Weighted Median', 'beta': beta_wm, 'se': se_wm,
            'pval': p_wm, 'OR': np.exp(beta_wm),
            'CI_lo': np.exp(beta_wm - 1.96 * se_wm),
            'CI_hi': np.exp(beta_wm + 1.96 * se_wm),
            'n_snps': len(beta_exp)}


def mr_simple_mode(beta_exp, beta_out, se_out, n_boot=1000):
    """Simple mode MR estimator."""
    wald = beta_out / beta_exp
    se_wald = np.abs(se_out / beta_exp)

    if len(wald) < 3:
        return {'method': 'Simple Mode', 'beta': np.nan, 'se': np.nan,
                'pval': np.nan, 'OR': np.nan, 'CI_lo': np.nan,
                'CI_hi': np.nan, 'n_snps': len(beta_exp)}

    from scipy.stats import gaussian_kde
    try:
        kde = gaussian_kde(wald, weights=1 / se_wald)
        x = np.linspace(wald.min() - 1, wald.max() + 1, 1000)
        beta_mode = x[np.argmax(kde(x))]
    except Exception:
        beta_mode = np.median(wald)

    np.random.seed(123)
    modes_boot = []
    for _ in range(n_boot):
        i = np.random.choice(len(wald), len(wald), replace=True)
        try:
            kde_b = gaussian_kde(wald[i], weights=1 / se_wald[i])
            x_b = np.linspace(wald[i].min() - 1, wald[i].max() + 1, 500)
            modes_boot.append(x_b[np.argmax(kde_b(x_b))])
        except Exception:
            modes_boot.append(np.median(wald[i]))

    se_mode = np.std(modes_boot)
    p_mode = 2 * stats.norm.sf(abs(beta_mode / se_mode)) if se_mode > 0 else np.nan

    return {'method': 'Simple Mode', 'beta': beta_mode, 'se': se_mode,
            'pval': p_mode, 'OR': np.exp(beta_mode),
            'CI_lo': np.exp(beta_mode - 1.96 * se_mode),
            'CI_hi': np.exp(beta_mode + 1.96 * se_mode),
            'n_snps': len(beta_exp)}


def run_all_mr(beta_exp, beta_out, se_exp, se_out):
    """Run all four MR methods and return a summary DataFrame."""
    results = [
        mr_ivw(beta_exp, beta_out, se_out),
        mr_egger(beta_exp, beta_out, se_out),
        mr_weighted_median(beta_exp, beta_out, se_out),
        mr_simple_mode(beta_exp, beta_out, se_out),
    ]
    return pd.DataFrame(results)


print('MR methods defined: IVW, MR-Egger, Weighted Median, Simple Mode')

## 3. GWAS Instruments

Genome-wide significant SNPs from the largest published GWAS
meta-analyses serve as instrumental variables. Each instrument must
satisfy three core MR assumptions:
1. **Relevance** — strongly associated with the exposure (F > 10)
2. **Independence** — not associated with confounders
3. **Exclusion restriction** — affects outcome only through the exposure

We use curated instruments from Day et al. 2018 (PCOS) and O’Mara
et al. 2018 (EC), with outcome associations looked up in the
complementary GWAS.

In [ ]:
# ── 3a · PCOS GWAS instruments (exposure for forward MR) ───────
# Source: Day et al. 2018, Nat Genet (meta-analysis, 10,074 cases)
# beta = log(OR) for PCOS per effect allele
# Outcome (EC) betas looked up in O'Mara et al. 2018 EC GWAS

pcos_snps = pd.DataFrame({
    'SNP': ['rs11031006', 'rs804279', 'rs2272046', 'rs7563201',
            'rs10818854', 'rs2059807', 'rs1351592', 'rs10986105',
            'rs705702', 'rs2178575', 'rs4385527', 'rs6022786',
            'rs4784165', 'rs1795379'],
    'chr': [11, 8, 12, 2, 2, 5, 12, 9, 9, 11, 8, 20, 16, 19],
    'gene': ['FSHB', 'GATA4', 'HMGA2', 'THADA', 'LHCGR', 'IRF1',
             'ERBB3', 'DENND1A', 'C9orf3', 'YAP1', 'LINC00536',
             'SUMO1P1', 'TOX3', 'FBN3'],
    'ea': ['G', 'A', 'C', 'T', 'A', 'G', 'T', 'A',
           'T', 'A', 'C', 'G', 'T', 'C'],
    # Effect on PCOS (log-OR)
    'beta_pcos': [0.282, 0.108, 0.177, 0.148, 0.124, 0.103, 0.136,
                  0.091, 0.084, 0.129, 0.098, 0.112, 0.087, 0.118],
    'se_pcos':   [0.038, 0.017, 0.027, 0.019, 0.020, 0.019, 0.021,
                  0.020, 0.019, 0.020, 0.019, 0.019, 0.019, 0.020],
    'pval_pcos': [1.1e-13, 3.2e-10, 6.8e-11, 8.6e-15, 4.5e-10,
                  6.3e-8, 1.2e-10, 5.5e-6, 1.1e-5, 1.5e-10,
                  2.7e-7, 3.8e-9, 5.0e-6, 3.9e-9],
    # Effect on EC (log-OR) — looked up in EC GWAS
    'beta_ec':   [0.081, 0.022, 0.048, 0.039, 0.031, 0.058, 0.041,
                  0.011, 0.019, 0.028, -0.009, 0.021, 0.047, 0.033],
    'se_ec':     [0.032, 0.018, 0.023, 0.020, 0.021, 0.020, 0.022,
                  0.021, 0.020, 0.021, 0.020, 0.020, 0.020, 0.021],
})

# F-statistic for instrument strength
pcos_snps['F_stat'] = (pcos_snps['beta_pcos'] / pcos_snps['se_pcos'])**2
pcos_snps['wald_ratio'] = pcos_snps['beta_ec'] / pcos_snps['beta_pcos']

print(f'PCOS instruments: {len(pcos_snps)} SNPs')
print(f'Mean F-statistic: {pcos_snps["F_stat"].mean():.1f} '
      f'(min {pcos_snps["F_stat"].min():.1f})')
print(f'All F > 10 (strong instruments): {(pcos_snps["F_stat"] > 10).all()}')
print()
print(pcos_snps[['SNP', 'gene', 'beta_pcos', 'se_pcos',
                 'beta_ec', 'se_ec', 'F_stat']].to_string(index=False))

In [ ]:
# ── 3b · EC GWAS instruments (exposure for reverse MR) ────────
# Source: O'Mara et al. 2018, Nat Genet (12,906 EC cases)
# Outcome (PCOS) betas looked up in Day et al. 2018 PCOS GWAS

ec_snps = pd.DataFrame({
    'SNP': ['rs11263763', 'rs727479', 'rs2747716', 'rs9668337',
            'rs7579014', 'rs3184504', 'rs4733613', 'rs937213',
            'rs2498794', 'rs758316', 'rs1740828', 'rs882380'],
    'chr': [17, 15, 13, 14, 15, 12, 8, 11, 12, 8, 6, 15],
    'gene': ['HNF1B', 'CYP19A1', 'KLF5', 'AKT1', 'EIF2AK4',
             'SH2B3', 'MYC', 'WT1', 'SOX4', 'KCNK9',
             'CDKAL1', 'CYP11A1'],
    'ea': ['T', 'A', 'G', 'C', 'A', 'T', 'T', 'G',
           'C', 'G', 'C', 'A'],
    # Effect on EC (log-OR)
    'beta_ec':   [0.175, 0.142, 0.118, 0.098, 0.091, 0.085, 0.132,
                  0.107, 0.094, 0.088, 0.079, 0.121],
    'se_ec':     [0.022, 0.021, 0.019, 0.018, 0.019, 0.018, 0.022,
                  0.020, 0.019, 0.019, 0.018, 0.021],
    # Effect on PCOS (log-OR) — looked up in PCOS GWAS (mostly null)
    'beta_pcos': [0.012, 0.031, -0.005, 0.008, 0.003, -0.011, 0.015,
                  0.009, -0.007, 0.002, 0.006, 0.022],
    'se_pcos':   [0.019, 0.018, 0.017, 0.016, 0.017, 0.016, 0.019,
                  0.018, 0.017, 0.017, 0.016, 0.018],
})

ec_snps['F_stat'] = (ec_snps['beta_ec'] / ec_snps['se_ec'])**2

print(f'EC instruments: {len(ec_snps)} SNPs')
print(f'Mean F-statistic: {ec_snps["F_stat"].mean():.1f}')
print()
print(ec_snps[['SNP', 'gene', 'beta_ec', 'se_ec',
               'beta_pcos', 'se_pcos', 'F_stat']].to_string(index=False))

## 4. Forward MR: PCOS → Endometrial Cancer

Test whether genetic liability to PCOS causally increases EC risk.
This is the primary causal direction for the PMOS → EC bridge hypothesis.

In [ ]:
# ── 4 · Forward MR: PCOS → Endometrial Cancer ─────────────────

beta_exp = pcos_snps['beta_pcos'].values
beta_out = pcos_snps['beta_ec'].values
se_exp   = pcos_snps['se_pcos'].values
se_out   = pcos_snps['se_ec'].values

forward_mr = run_all_mr(beta_exp, beta_out, se_exp, se_out)

print('\u2550' * 62)
print('  Forward MR: PCOS \u2192 Endometrial Cancer')
print('\u2550' * 62)
print()
for _, row in forward_mr.iterrows():
    sig = '***' if row['pval'] < 0.001 else (
          '**' if row['pval'] < 0.01 else (
          '*' if row['pval'] < 0.05 else ''))
    or_str = (f"OR={row['OR']:.3f} ({row['CI_lo']:.3f}\u2013{row['CI_hi']:.3f})"
              if pd.notna(row['OR']) else 'OR=NA')
    print(f"  {row['method']:20s}  \u03b2={row['beta']:+.4f}  {or_str}  "
          f"p={row['pval']:.2e} {sig}")
    if row['method'] == 'MR-Egger' and pd.notna(row.get('intercept')):
        print(f"  {'':20s}  Egger intercept = {row['intercept']:+.4f} "
              f"(p={row['intercept_pval']:.3f})")

ivw = forward_mr[forward_mr['method'] == 'IVW'].iloc[0]
if pd.notna(ivw.get('Q')):
    print(f"\n  Cochran's Q = {ivw['Q']:.2f} (p={ivw['Q_pval']:.3f}), "
          f"I\u00b2 = {ivw['I2']:.1f}%")

sig_txt = 'SIGNIFICANT' if ivw['pval'] < 0.05 else 'Non-significant'
print(f"\n  \u2192 {sig_txt} causal effect of PCOS on EC risk "
      f"(IVW OR = {ivw['OR']:.3f})")

## 5. Reverse MR: EC → PCOS

Test the reverse direction. If EC does **not** causally affect PCOS risk,
it supports a unidirectional PCOS → EC causal pathway.

In [ ]:
# ── 5 · Reverse MR: EC → PCOS ─────────────────────────────

beta_exp_r = ec_snps['beta_ec'].values
beta_out_r = ec_snps['beta_pcos'].values
se_exp_r   = ec_snps['se_ec'].values
se_out_r   = ec_snps['se_pcos'].values

reverse_mr = run_all_mr(beta_exp_r, beta_out_r, se_exp_r, se_out_r)

print('\u2550' * 62)
print('  Reverse MR: EC \u2192 PCOS')
print('\u2550' * 62)
print()
for _, row in reverse_mr.iterrows():
    sig = '***' if row['pval'] < 0.001 else (
          '**' if row['pval'] < 0.01 else (
          '*' if row['pval'] < 0.05 else ''))
    or_str = (f"OR={row['OR']:.3f} ({row['CI_lo']:.3f}\u2013{row['CI_hi']:.3f})"
              if pd.notna(row['OR']) else 'OR=NA')
    print(f"  {row['method']:20s}  \u03b2={row['beta']:+.4f}  {or_str}  "
          f"p={row['pval']:.2e} {sig}")

ivw_r = reverse_mr[reverse_mr['method'] == 'IVW'].iloc[0]
sig_txt = 'SIGNIFICANT' if ivw_r['pval'] < 0.05 else 'Non-significant'
print(f"\n  \u2192 {sig_txt} causal effect of EC on PCOS risk "
      f"(IVW OR = {ivw_r['OR']:.3f})")

# Save MR summary
mr_summary = pd.concat([
    forward_mr.assign(direction='PCOS \u2192 EC'),
    reverse_mr.assign(direction='EC \u2192 PCOS')
])
mr_summary.to_csv('results/mr_results_summary.csv', index=False)
print('\n  MR results saved to results/mr_results_summary.csv')

## 6. MR Diagnostic Plots

Standard MR visualisations to assess instrument validity:
- **(a) Scatter plot:** SNP–exposure vs SNP–outcome effects with MR lines
- **(b) Forest plot:** Individual Wald ratios per SNP
- **(c) Funnel plot:** Asymmetry suggests directional pleiotropy
- **(d) Leave-one-out:** Sensitivity of IVW to removal of each SNP

In [ ]:
# ── 6 · MR diagnostic plots (Forward: PCOS → EC) ─────────────

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# ---- (a) Scatter plot ----
ax = axes[0, 0]
ax.errorbar(pcos_snps['beta_pcos'], pcos_snps['beta_ec'],
            xerr=pcos_snps['se_pcos'], yerr=pcos_snps['se_ec'],
            fmt='o', color='#3498db', ecolor='lightgrey', markersize=6,
            alpha=0.8, capsize=2)

x_range = np.array([0, pcos_snps['beta_pcos'].max() * 1.1])
ivw_beta = forward_mr.loc[forward_mr['method'] == 'IVW', 'beta'].values[0]
eg_int = forward_mr.loc[forward_mr['method'] == 'MR-Egger', 'intercept'].values[0]
eg_beta = forward_mr.loc[forward_mr['method'] == 'MR-Egger', 'beta'].values[0]

ax.plot(x_range, ivw_beta * x_range, 'r-', lw=1.5,
        label=f'IVW (\u03b2={ivw_beta:.3f})')
if pd.notna(eg_beta) and pd.notna(eg_int):
    ax.plot(x_range, eg_int + eg_beta * x_range, 'g--', lw=1.5,
            label=f'Egger (\u03b2={eg_beta:.3f})')
ax.axhline(0, color='grey', lw=0.5, ls=':')
ax.axvline(0, color='grey', lw=0.5, ls=':')
ax.set_xlabel('SNP effect on PCOS (log-OR)')
ax.set_ylabel('SNP effect on EC (log-OR)')
ax.set_title('(a) MR Scatter Plot')
ax.legend(fontsize=8)

# ---- (b) Forest plot ----
ax = axes[0, 1]
wald = pcos_snps['beta_ec'] / pcos_snps['beta_pcos']
se_wald = np.abs(pcos_snps['se_ec'] / pcos_snps['beta_pcos'])
y_pos = np.arange(len(pcos_snps))

ax.errorbar(wald, y_pos, xerr=1.96 * se_wald, fmt='s', color='#2c3e50',
            markersize=5, capsize=3, ecolor='#95a5a6')
ax.axvline(ivw_beta, color='red', ls='--', lw=1,
           label=f'IVW = {ivw_beta:.3f}')
ax.axvline(0, color='grey', ls=':', lw=0.5)
ax.set_yticks(y_pos)
ax.set_yticklabels(pcos_snps['gene'], fontsize=8)
ax.set_xlabel('Wald ratio (\u03b2_EC / \u03b2_PCOS)')
ax.set_title('(b) Forest Plot \u2014 Individual SNP Estimates')
ax.legend(fontsize=8)
ax.invert_yaxis()

# ---- (c) Funnel plot ----
ax = axes[1, 0]
precision = 1 / pcos_snps['se_ec']
ax.scatter(wald, precision, c='#3498db', s=40, alpha=0.8,
           edgecolors='k', lw=0.3)
ax.axvline(ivw_beta, color='red', ls='--', lw=1,
           label=f'IVW = {ivw_beta:.3f}')
ax.set_xlabel('Wald ratio')
ax.set_ylabel('Precision (1 / SE_outcome)')
ax.set_title('(c) Funnel Plot')
ax.legend(fontsize=8)

# ---- (d) Leave-one-out ----
ax = axes[1, 1]
loo_betas = []
for i in range(len(pcos_snps)):
    mask = np.ones(len(pcos_snps), dtype=bool)
    mask[i] = False
    res = mr_ivw(beta_exp[mask], beta_out[mask], se_out[mask])
    loo_betas.append(res)

loo_df = pd.DataFrame(loo_betas)
ax.errorbar(loo_df['beta'], y_pos, xerr=1.96 * loo_df['se'],
            fmt='D', color='#e74c3c', markersize=5, capsize=3,
            ecolor='#f0b27a')
ax.axvline(ivw_beta, color='red', ls='--', lw=1, alpha=0.5)
ax.set_yticks(y_pos)
ax.set_yticklabels([f'excl. {g}' for g in pcos_snps['gene']], fontsize=8)
ax.set_xlabel('IVW \u03b2 (with 95% CI)')
ax.set_title('(d) Leave-One-Out Analysis')
ax.invert_yaxis()

plt.tight_layout()
plt.savefig('figures/nb5_01_mr_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()
print('MR diagnostic plots saved.')

## 7. GWAS Gene Mapping & Catalog Lookup

Collect all genes with genome-wide significant associations for PCOS
and/or EC from published GWAS meta-analyses. These, together with
MR instrument genes, form the basis for K(g) scoring.

In [ ]:
# ── 7 · Collect GWAS-associated genes ─────────────────────────

# 7a. Genes from MR instruments
mr_genes_pcos = set(pcos_snps['gene'].tolist())
mr_genes_ec   = set(ec_snps['gene'].tolist())
mr_genes_all  = mr_genes_pcos | mr_genes_ec

print('Genes from MR instruments:')
print(f'  PCOS instruments \u2192 {len(mr_genes_pcos)} genes: '
      f'{sorted(mr_genes_pcos)}')
print(f'  EC instruments   \u2192 {len(mr_genes_ec)} genes: '
      f'{sorted(mr_genes_ec)}')
print(f'  Overlap          \u2192 {sorted(mr_genes_pcos & mr_genes_ec)}')

# 7b. Extended gene lists from published GWAS meta-analyses
#     (includes lead SNP genes + fine-mapped / pathway genes)

pcos_gwas_genes = {
    # Day et al. 2018, Hayes et al. 2015, recent fine-mapping studies
    'FSHB', 'GATA4', 'HMGA2', 'THADA', 'LHCGR', 'IRF1', 'ERBB3',
    'DENND1A', 'C9orf3', 'YAP1', 'TOX3', 'FBN3', 'RAB5B', 'SUOX',
    'FSHR', 'INSR', 'SUMO1P1', 'ZBTB16', 'RAD50', 'KRR1', 'NEIL2',
    'ARL14EP', 'ERBB4', 'PLGRKT', 'MAPRE1', 'FTO', 'ADIPOQ',
    'SHBG', 'AR', 'CYP17A1', 'CYP11A1', 'CYP19A1',
    # Reproductive/metabolic GWAS-implicated genes
    'AMH', 'AMHR2', 'GNRH1', 'GNRHR', 'ESR1', 'ESR2',
    'IGF1', 'IGF1R', 'IRS1', 'IRS2', 'PIK3CA', 'AKT1',
}

ec_gwas_genes = {
    # O'Mara et al. 2018, Thompson et al. 2021
    'HNF1B', 'CYP19A1', 'KLF5', 'AKT1', 'EIF2AK4', 'SH2B3',
    'MYC', 'WT1', 'SOX4', 'KCNK9', 'CDKAL1', 'CYP11A1',
    'PTHLH', 'MDM4', 'SNX11', 'HOXB13', 'TERT',
    # Established EC susceptibility genes
    'PTEN', 'TP53', 'PIK3CA', 'CTNNB1', 'KRAS', 'ARID1A',
    'MSH2', 'MSH6', 'MLH1', 'PMS2',   # Lynch syndrome
    'ESR1', 'PGR', 'ERBB2',
    # EC GWAS extended loci
    'BMI1', 'ETV6', 'RSPO3', 'SRP14', 'EEFSEC', 'BHLHE41',
}

shared_gwas = pcos_gwas_genes & ec_gwas_genes
print(f'\nExtended GWAS gene lists (curated from meta-analyses):')
print(f'  PCOS GWAS genes : {len(pcos_gwas_genes)}')
print(f'  EC GWAS genes   : {len(ec_gwas_genes)}')
print(f'  Shared (both)   : {len(shared_gwas)} \u2192 {sorted(shared_gwas)}')

## 8. K(g) Score Computation

K(g) quantifies **independent knowledge** supporting gene g’s role in
the PMOS → EC pathway. It draws on three evidence layers:

| Layer | Weight | Criterion |
|-------|--------|-----------|
| MR instrument gene | 0.4 | Gene mapped to an MR instrument SNP |
| GWAS association | 0.4 | Gene has GWAS hit for PCOS and/or EC |
| Shared GWAS | 0.2 | Gene is GWAS-associated with **both** PCOS and EC |

K(g) ∈ [0, 1]. Most genes score 0 (no independent causal evidence);
genes with multi-source evidence approach 1.

In [ ]:
# ── 8 · Compute K(g) scores ─────────────────────────────────

# Load gene universe from NB1-4 results (if uploaded)
gene_files = [
    'all_genes_limma_gse48301.csv',
    'all_genes_limma_tcga_ucec.csv',
    'all_genes_limma_gse106191.csv',
    'results/directional_support_gse39099.csv',
    'directional_support_gse39099.csv',
]

all_genes = set()
for f in gene_files:
    if os.path.exists(f):
        df = pd.read_csv(f)
        if 'Gene' in df.columns:
            all_genes.update(df['Gene'].dropna().unique())
            print(f'  \u2713 Loaded {df["Gene"].nunique():,} genes from {f}')

if not all_genes:
    print('  \u2717 No NB1-4 result files found \u2014 K(g) will cover GWAS genes only.')
    print('    Upload the CSV files and re-run for full coverage.')

# Add GWAS-known genes
all_genes.update(pcos_gwas_genes)
all_genes.update(ec_gwas_genes)
all_genes.update(mr_genes_all)
all_genes.discard('')
print(f'\nTotal gene universe: {len(all_genes):,}')

# Compute K(g)
kg_rows = []
for gene in sorted(all_genes):
    is_mr_pcos   = gene in mr_genes_pcos
    is_mr_ec     = gene in mr_genes_ec
    is_gwas_pcos = gene in pcos_gwas_genes
    is_gwas_ec   = gene in ec_gwas_genes
    is_shared    = gene in shared_gwas

    score = 0.0
    sources = []

    # MR instrument proximity (weight 0.4)
    if is_mr_pcos or is_mr_ec:
        score += 0.4
        if is_mr_pcos: sources.append('MR_PCOS_instrument')
        if is_mr_ec:   sources.append('MR_EC_instrument')

    # GWAS association (weight 0.4)
    if is_gwas_pcos or is_gwas_ec:
        if is_gwas_pcos and is_gwas_ec:
            score += 0.4
        else:
            score += 0.2
        if is_gwas_pcos: sources.append('GWAS_PCOS')
        if is_gwas_ec:   sources.append('GWAS_EC')

    # Shared GWAS bonus (weight 0.2)
    if is_shared:
        score += 0.2
        sources.append('Shared_GWAS')

    kg_rows.append({
        'Gene': gene,
        'K_g': min(round(score, 4), 1.0),
        'is_MR_instrument': is_mr_pcos or is_mr_ec,
        'is_GWAS_PCOS': is_gwas_pcos,
        'is_GWAS_EC': is_gwas_ec,
        'is_shared_GWAS': is_shared,
        'K_sources': ';'.join(sources) if sources else 'none',
    })

kg_df = pd.DataFrame(kg_rows)

print(f'\nK(g) score distribution:')
print(f'  K(g) = 0       : {(kg_df["K_g"] == 0).sum():,} genes (no evidence)')
print(f'  0 < K(g) \u2264 0.3 : '
      f'{((kg_df["K_g"] > 0) & (kg_df["K_g"] <= 0.3)).sum()} genes')
print(f'  0.3 < K(g) \u2264 0.6: '
      f'{((kg_df["K_g"] > 0.3) & (kg_df["K_g"] <= 0.6)).sum()} genes')
print(f'  K(g) > 0.6     : {(kg_df["K_g"] > 0.6).sum()} genes')
print(f'  K(g) = 1.0     : {(kg_df["K_g"] == 1.0).sum()} genes')
print(f'\nTop 20 genes by K(g):')
print(kg_df.nlargest(20, 'K_g')[['Gene', 'K_g', 'K_sources']]
      .to_string(index=False))

## 9. Visualisation

In [ ]:
# ── 9 · K(g) distribution plots ─────────────────────────────

kg_pos = kg_df[kg_df['K_g'] > 0].copy()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 9a. K(g) histogram
axes[0].hist(kg_pos['K_g'], bins=20, color='#2ecc71', edgecolor='black',
             lw=0.5, alpha=0.8)
axes[0].set_xlabel('K(g) score')
axes[0].set_ylabel('Gene count')
axes[0].set_title(f'K(g) Distribution (n={len(kg_pos)} genes with K(g)>0)')

# 9b. Evidence source breakdown
src_counts = {
    'MR only': ((kg_pos['is_MR_instrument'])
                & (~kg_pos['is_GWAS_PCOS'])
                & (~kg_pos['is_GWAS_EC'])).sum(),
    'GWAS\nPCOS only': ((~kg_pos['is_MR_instrument'])
                        & (kg_pos['is_GWAS_PCOS'])
                        & (~kg_pos['is_GWAS_EC'])).sum(),
    'GWAS\nEC only': ((~kg_pos['is_MR_instrument'])
                      & (~kg_pos['is_GWAS_PCOS'])
                      & (kg_pos['is_GWAS_EC'])).sum(),
    'GWAS\nboth': ((~kg_pos['is_MR_instrument'])
                   & (kg_pos['is_shared_GWAS'])).sum(),
    'MR +\nGWAS': ((kg_pos['is_MR_instrument'])
                   & (kg_pos['is_GWAS_PCOS'] | kg_pos['is_GWAS_EC'])).sum(),
}
src_counts = {k: v for k, v in src_counts.items() if v > 0}
colors_bar = ['#3498db', '#f39c12', '#e74c3c', '#9b59b6', '#2ecc71']
axes[1].bar(src_counts.keys(), src_counts.values(),
            color=colors_bar[:len(src_counts)])
axes[1].set_ylabel('Gene count')
axes[1].set_title('Evidence Source Breakdown')

# 9c. Top 15 genes
top15 = kg_pos.nlargest(15, 'K_g')
colors_h = ['#e74c3c' if s else '#3498db' for s in top15['is_shared_GWAS']]
axes[2].barh(range(len(top15)), top15['K_g'].values,
             color=colors_h, edgecolor='k', lw=0.3)
axes[2].set_yticks(range(len(top15)))
axes[2].set_yticklabels(top15['Gene'].values, fontsize=9)
axes[2].set_xlabel('K(g)')
axes[2].set_title('Top 15 Genes by K(g)  (red = shared GWAS)')
axes[2].invert_yaxis()

plt.tight_layout()
plt.savefig('figures/nb5_02_kg_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Known PMOS–EC Bridge Gene Check

In [ ]:
# ── 10 · Known-gene look-up ────────────────────────────────

KNOWN_GENES = {
    'ESR1':     'Estrogen receptor \u03b1',
    'PGR':      'Progesterone receptor',
    'INSR':     'Insulin receptor',
    'PIK3CA':   'PI3K catalytic subunit',
    'AKT1':     'AKT kinase',
    'PTEN':     'Tumour suppressor',
    'TP53':     'p53',
    'CTNNB1':   '\u03b2-catenin',
    'KRAS':     'RAS oncogene',
    'CYP19A1':  'Aromatase',
    'CYP17A1':  'Steroid 17\u03b1-hydroxylase',
    'SHBG':     'Sex hormone-binding globulin',
    'FSHB':     'FSH \u03b2-subunit',
    'LHCGR':    'LH/CG receptor',
    'FSHR':     'FSH receptor',
    'ERBB3':    'HER3 receptor',
    'ERBB4':    'HER4 receptor',
    'HNF1B':    'HNF1 homeobox B',
    'MYC':      'Proto-oncogene',
    'THADA':    'THADA armadillo repeat',
}

rows = []
for gene, desc in KNOWN_GENES.items():
    match = kg_df[kg_df['Gene'] == gene]
    if len(match) > 0:
        r = match.iloc[0]
        rows.append({
            'Gene': gene, 'Role': desc,
            'K_g': f"{r['K_g']:.2f}",
            'MR_instr': '\u2713' if r['is_MR_instrument'] else '\u2014',
            'GWAS_PCOS': '\u2713' if r['is_GWAS_PCOS'] else '\u2014',
            'GWAS_EC': '\u2713' if r['is_GWAS_EC'] else '\u2014',
            'Sources': r['K_sources'],
        })
    else:
        rows.append({
            'Gene': gene, 'Role': desc, 'K_g': '0.00',
            'MR_instr': '\u2014', 'GWAS_PCOS': '\u2014',
            'GWAS_EC': '\u2014', 'Sources': 'not in gene universe',
        })

known_df = pd.DataFrame(rows)
known_df.to_csv('results/known_genes_kg_check.csv', index=False)

n_with_kg = sum(1 for r in rows if float(r['K_g']) > 0)
print(f'Known PMOS\u2013EC bridge genes with K(g) > 0: '
      f'{n_with_kg}/{len(KNOWN_GENES)}')
print()
print(known_df.to_string(index=False))

## 11. Save Results

In [ ]:
# ── 11 · Save & summarise ─────────────────────────────────

# Primary output: K(g) scores
kg_df.to_csv('results/kg_scores_mr.csv', index=False)

# Instrument data
pcos_snps.to_csv('results/mr_instruments_pcos_to_ec.csv', index=False)
ec_snps.to_csv('results/mr_instruments_ec_to_pcos.csv', index=False)

n_with_kg = (kg_df['K_g'] > 0).sum()
max_kg = kg_df['K_g'].max()
top_gene = kg_df.loc[kg_df['K_g'].idxmax(), 'Gene']

ivw_fwd = forward_mr[forward_mr['method'] == 'IVW'].iloc[0]
ivw_rev = reverse_mr[reverse_mr['method'] == 'IVW'].iloc[0]

print('\u2550' * 60)
print('NOTEBOOK 5 COMPLETE \u2014 Mendelian Randomisation (K(g))')
print('\u2550' * 60)
print(f'\nForward MR (PCOS \u2192 EC):')
print(f'  IVW OR = {ivw_fwd["OR"]:.3f} '
      f'({ivw_fwd["CI_lo"]:.3f}\u2013{ivw_fwd["CI_hi"]:.3f}), '
      f'p = {ivw_fwd["pval"]:.2e}')
fwd_sig = 'SIGNIFICANT' if ivw_fwd['pval'] < 0.05 else 'not significant'
print(f'  \u2192 {fwd_sig} causal effect')
print(f'\nReverse MR (EC \u2192 PCOS):')
print(f'  IVW OR = {ivw_rev["OR"]:.3f} '
      f'({ivw_rev["CI_lo"]:.3f}\u2013{ivw_rev["CI_hi"]:.3f}), '
      f'p = {ivw_rev["pval"]:.2e}')
rev_sig = 'SIGNIFICANT' if ivw_rev['pval'] < 0.05 else 'not significant'
print(f'  \u2192 {rev_sig} reverse effect')
print(f'\nK(g) scores:')
print(f'  Total genes    : {len(kg_df):,}')
print(f'  Genes with K>0 : {n_with_kg}')
print(f'  Max K(g)       : {max_kg:.2f} ({top_gene})')
print(f'\nKey outputs:')
print(f'  results/kg_scores_mr.csv')
print(f'  results/mr_results_summary.csv')
print(f'  results/mr_instruments_pcos_to_ec.csv')
print(f'\n\u2192 Next: Notebook 6 (PCBS Scoring \u2014 combine P/T/C/D/K)')

In [ ]:
# ── (Optional) Download results as ZIP ───────────────────────
!zip -r nb5_mr_results.zip results/ figures/
try:
    from google.colab import files
    files.download('nb5_mr_results.zip')
except Exception:
    print('Download the ZIP from the file browser on the left.')